# Starting with splitting data

In [1]:
from pathlib import Path
import sys
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, PandasTools, MolStandardize
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG=True
import rdkit.rdBase as rkrb
import rdkit.RDLogger as rkl
from IPython.display import display, display_png
from sklearn.preprocessing import FunctionTransformer
from pandas import DataFrame
import pandas as pd
import numpy as np
import logging
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import pickle
import random
from rdkit import DataStructs
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegressionCV, LinearRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import KFold, cross_val_predict, GridSearchCV
import sklearn.metrics as metrics
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, roc_auc_score, recall_score, precision_score
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.datasets import make_classification
from tqdm import tqdm
#import xgboost
#from xgboost import XGBRFClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
#import deepchem as dc
from rdkit.Chem import rdFingerprintGenerator


In [2]:
#!pip install mordred

In [3]:
#!pip install pandarallel

In [36]:
mchr1_united = pd.read_csv("chembl_set_sos1_big.csv")

/tmp/ipykernel_471808/1440510682.py:1: DtypeWarning: Columns (2,8,10,11,12,20,21,22,24,25,26,27,28,29,30,31,32,33,34,38,39,41,42,44,45,46,47,48,49,51,52,53,54,56,57,58,59,60,61,644) have mixed types. Specify dtype option on import or set low_memory=False.
  mchr1_united = pd.read_csv("chembl_set_sos1_big.csv")


In [37]:
len(mchr1_united)

5713

In [38]:
mchr1_united['class'].value_counts()

class
1    5450
0     263
Name: count, dtype: int64

In [8]:
selected_30 = PandasTools.LoadSDF('chembl_random_set.sdf',molColName='ROMol')

[21:58:47] Explicit valence for atom # 36 N, 4, is greater than permitted
[21:58:47] ERROR: Could not sanitize molecule ending on line 7298
[21:58:47] ERROR: Explicit valence for atom # 36 N, 4, is greater than permitted
[21:58:47] Explicit valence for atom # 8 O, 3, is greater than permitted
[21:58:47] ERROR: Could not sanitize molecule ending on line 14063
[21:58:47] ERROR: Explicit valence for atom # 8 O, 3, is greater than permitted
[21:58:47] Explicit valence for atom # 24 O, 3, is greater than permitted
[21:58:47] ERROR: Could not sanitize molecule ending on line 45246
[21:58:47] ERROR: Explicit valence for atom # 24 O, 3, is greater than permitted
[21:58:48] Explicit valence for atom # 9 O, 3, is greater than permitted
[21:58:48] ERROR: Could not sanitize molecule ending on line 49159
[21:58:48] ERROR: Explicit valence for atom # 9 O, 3, is greater than permitted
[21:58:48] Explicit valence for atom # 12 O, 3, is greater than permitted
[21:58:48] ERROR: Could not sanitize molecu

In [39]:
mchr1_united = mchr1_united[['canonical_smiles', 'standard_units', 'standard_value', 'class', 'ID']]

In [40]:
mchr1_united = mchr1_united[['canonical_smiles', 'standard_value', 'class', 'ID']]

In [ ]:
selected_30.columns = ['canonical_smiles', 'standard_type', 'value_nM', 'class', 'molecule_chembl_id']

In [42]:
#mchr1_united = mchr1_united[['canonical_smiles', 'standard_type', 'standard_value', 'class', 'ID']]
selected_30 = selected_30.rename(columns={'molecule_chembl_id': 'ID'})
mchr1_united = mchr1_united.rename(columns={'standard_value': 'value_nM'})

In [43]:
selected_30 = selected_30[['canonical_smiles', 'value_nM', 'class', 'ID']]

In [44]:
selected_30 = selected_30.sample(frac=1).reset_index(drop=True)

In [45]:
n_subsets = 10
subsets = [selected_30.iloc[i::n_subsets].reset_index(drop=True) for i in range(n_subsets)]
subsets[0]

,canonical_smiles,value_nM,class,ID
0,CCOC(=O)C1=C(C)NC(=O)N(CCC(=O)NNC(=S)Nc2ccccc2...,Random,0,test_outputs/chembl_inactive_pdb/molecule_2984...
1,CC(C)(C)OC(=O)N[C@H]1COCCC/C=C\[C@@H]2C[C@@]2(...,Random,0,test_outputs/chembl_inactive_pdb/molecule_2736...
2,CS(=O)(=O)N(Cc1ccc(C(=O)NN=C2CCCC2)cc1)c1ccccc1,Random,0,test_outputs/chembl_inactive_pdb/molecule_1525...
3,COC(=O)c1cn(NC(=O)c2cc([N+](=O)[O-])ccc2F)c(=O...,Random,0,test_outputs/chembl_inactive_pdb/molecule_2572...
4,CC1(C)C(=O)OC(/C=C\c2cccc([N+](=O)[O-])c2)C(C)...,Random,0,test_outputs/chembl_inactive_pdb/molecule_2121...
...,...,...,...,...
2940,CC(C)Nc1ccc2cnn(-c3cncc(-c4csc(C(=O)O)c4)n3)c2c1,Random,0,test_outputs/chembl_inactive_pdb/molecule_3384...
2941,CO[C@H]1CC[C@H](CC(=O)N[C@H]2CC[C@H](CCN3CCC(c...,Random,0,test_outputs/chembl_inactive_pdb/molecule_7353...
2942,COc1cc2nccc(Oc3ccc(NC(=O)c4c(N5CCN(C)CC5)ccn(-...,Random,0,test_outputs/chembl_inactive_pdb/molecule_1303...
2943,CN1C(=O)c2c(ncn2CC(=O)Nc2nc(-c3ccc(C(C)(C)C)cc...,Random,0,test_outputs/chembl_inactive_pdb/molecule_2906...


In [46]:
subsets = [pd.concat([mchr1_united, i], ignore_index=True) for i in subsets]

In [47]:
subsets[0]

,canonical_smiles,value_nM,class,ID
0,CC[C@H](C)[C@H](NC(=O)CN1C/C=C\CCC(=O)N[C@@H](...,25000.0,0,CHEMBL2086797
1,N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N1CCC2(CC1)CN(C...,37300.0,0,CHEMBL4160864
2,N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N1CC2(CCN(Cc3c[...,100000.0,0,CHEMBL4168801
3,N[C@@H](Cc1c[nH]c2ccccc12)C(=O)NCC1CCN(Cc2c[nH...,13300.0,0,CHEMBL4171085
4,N[C@@H](Cc1c[nH]c2cc(Cl)ccc12)C(=O)NC1CCN(Cc2c...,1400.0,1,CHEMBL4160484
...,...,...,...,...
8653,CC(C)Nc1ccc2cnn(-c3cncc(-c4csc(C(=O)O)c4)n3)c2c1,Random,0,test_outputs/chembl_inactive_pdb/molecule_3384...
8654,CO[C@H]1CC[C@H](CC(=O)N[C@H]2CC[C@H](CCN3CCC(c...,Random,0,test_outputs/chembl_inactive_pdb/molecule_7353...
8655,COc1cc2nccc(Oc3ccc(NC(=O)c4c(N5CCN(C)CC5)ccn(-...,Random,0,test_outputs/chembl_inactive_pdb/molecule_1303...
8656,CN1C(=O)c2c(ncn2CC(=O)Nc2nc(-c3ccc(C(C)(C)C)cc...,Random,0,test_outputs/chembl_inactive_pdb/molecule_2906...


In [48]:
with open('subsets_10_big.pkl', 'wb') as file:
    pickle.dump(subsets, file)

In [49]:
with open('subsets_10_big.pkl', 'rb') as file:
    subsets = pickle.load(file)

In [50]:
selected_30.columns

Index(['canonical_smiles', 'value_nM', 'class', 'ID'], dtype='object')

In [51]:
mchr1_united.columns

Index(['canonical_smiles', 'value_nM', 'class', 'ID'], dtype='object')

# Training on the WHOLE set

In [52]:
#! pip install ipywidgets

In [53]:
set = pd.concat([mchr1_united, selected_30], ignore_index=True)

In [54]:
len(set)

35157

In [60]:
set['class'] = set['class'].astype(int)

In [61]:
set['Scaffold'] = set['canonical_smiles'].apply(lambda x: Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(x))))

In [62]:
def train_test_onescaffold_split(df, test_size1 = 0.2):
    grouped = df.groupby('Scaffold')
    shuffled_groups = grouped.groups.keys()
    shuffled_groups = list(shuffled_groups)
    random.shuffle(shuffled_groups)
    train_scaffolds, test_scaffolds = train_test_split(shuffled_groups, test_size=test_size1, random_state=42)
    train_mask = df['Scaffold'].isin(train_scaffolds)
    test_mask = df['Scaffold'].isin(test_scaffolds)
    train_df = df[train_mask]
    test_df = df[test_mask]
    return train_df, test_df

In [63]:
train_df, test_df = train_test_onescaffold_split(set)

In [64]:
train_df['class'].value_counts()

class
0    23764
1     4451
Name: count, dtype: int64

In [65]:
test_df['class'].value_counts()

class
0    5943
1     999
Name: count, dtype: int64

# Features calculation

In [66]:
def calc_morgan_fingerprints(mol, r=2, nBits=512, errors_as_zeros=True):
    try:
        fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=r, fpSize=nBits)
        fp = fpgen.GetFingerprint(mol)
        arr = np.zeros((nBits,), dtype=np.int32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.astype(np.float32)
    except:
        return np.NaN if not errors_as_zeros else np.zeros((nBits,), dtype=np.float32)

In [67]:
from mordred import Calculator, descriptors, is_missing
calc = Calculator(descriptors, ignore_3D=False)

In [68]:

def calc_mordred_descriptors(mol, l=1826, errors_as_zeros=True):
    try:
        result = calc(mol)
        desc_list = [r if not is_missing(r) else 0 for r in result]
        np_arr = np.array(desc_list)
        return np_arr
    except:
        return np.NaN if not errors_as_zeros else np.zeros((l,), dtype=np.float32)

In [69]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys, DataStructs
import numpy as np

In [70]:
def calc_MACCS_fingerprints(mol, nBits=167, errors_as_zeros=True):
    try:
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((nBits,), dtype=np.int32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr.astype(np.float32)
    except:
        return np.NaN if not errors_as_zeros else np.zeros((nBits,), dtype=np.float32)

In [71]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 6 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [72]:
from tqdm.auto import tqdm
tqdm.pandas()

In [73]:
train_df['ROMol'] = train_df['canonical_smiles'].progress_apply(Chem.MolFromSmiles)

  0%|          | 0/28215 [00:00<?, ?it/s]

/tmp/ipykernel_471808/2559692629.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['ROMol'] = train_df['canonical_smiles'].progress_apply(Chem.MolFromSmiles)


In [74]:
train_df['MorganFP'] = train_df['ROMol'].parallel_apply(calc_morgan_fingerprints)

/tmp/ipykernel_471808/4049408413.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['MorganFP'] = train_df['ROMol'].parallel_apply(calc_morgan_fingerprints)


In [75]:
train_df['MACCSFP'] = train_df['ROMol'].parallel_apply(calc_MACCS_fingerprints)


/tmp/ipykernel_471808/753497247.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['MACCSFP'] = train_df['ROMol'].parallel_apply(calc_MACCS_fingerprints)


In [76]:
sum(train_df['MorganFP']) # Review that Morgan FP vector is not null

array([ 4391.,  9543.,  3894.,  3516.,  8213.,  2542.,  1463.,  1364.,
        2902.,  1573.,  2882.,  4344.,  1408.,  2379.,  3107.,  7640.,
        1718.,   816.,   844.,  3324.,  1209.,  1900.,  1456.,  1242.,
        1074.,  2051.,  1590.,  1089.,  2717.,  4222.,   710.,  2759.,
        1467., 20297.,  1690.,  1033.,  7760.,  3868.,  1514.,  2897.,
        3183.,  3250.,  5871.,  2043.,  1655.,  4091.,  3870.,  1004.,
         707.,  5300.,  3083.,   779.,  1639.,  3286.,   780.,   678.,
        1905.,   701.,  1868.,  1318.,   856.,  3189.,  1789.,  3166.,
       18975.,  2648.,  2605.,  3063.,  2591.,  2707.,   630.,  2017.,
        2514.,  5937.,  2984.,  2043.,  2219.,  3275.,  1080.,  2237.,
       18807.,  1751.,  1566.,  1590.,  1943.,  1086.,  1957.,   975.,
        1632.,   605.,  8812.,  3859.,  3057.,   792.,  3918.,  2324.,
        1084.,  1820.,  2225.,  1268.,  1402.,  1397.,  5396.,   788.,
        2479.,  1104.,  1182.,  1285.,  1822.,  2132.,  1836.,  3423.,
      

In [77]:
from scipy.sparse import csr_matrix

In [78]:
X_train = csr_matrix(np.vstack(train_df['MorganFP'].values))

In [79]:
y_train = train_df['class'].values

In [80]:
with open('fps_morgans_big.pkl', 'wb') as file:
    pickle.dump(X_train, file)

In [109]:
# ! pip install xgboost

In [110]:
 #! pip install optuna

In [81]:
import xgboost as xgb

In [82]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.num_row(), dtrain.num_col()

(28215, 512)

In [83]:
prevalence = train_df['class'].astype(int).sum() / train_df['class'].astype(int).shape[0]
print(f'prevalence is {prevalence:.5f}')

prevalence is 0.15775


In [84]:
import optuna


def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 6)
    gamma = trial.suggest_loguniform('gamma', 0.01, 10)
    colsample_bytree = trial.suggest_uniform('colsample_bytree', 0.1, 0.9)

    params = {'max_depth': max_depth,
          'eta': 0.05,
          'objective': 'binary:logistic',
          'eval_metric':['auc', 'map', 'logloss'], # acc, bac, f1_score, precission and recall are not supported in eval_metrics
          'tree_method': 'hist',
          'gamma': gamma,
          'colsample_bytree': colsample_bytree,
          'subsample': 0.5,
          'nthread': 8,
          'base_score': prevalence}

    res = xgb.cv(params,
                 dtrain,
                 num_boost_round=2000,
                 verbose_eval=False,
                 early_stopping_rounds=20,
                 nfold=5,
                 seed=113,
                 shuffle=True)

    metric = res['test-map-mean'].max()
    return metric

In [85]:
study_name = 'xgboost_fingerprints_big'
study = optuna.create_study(study_name=study_name,
                          storage=f'sqlite:///{study_name}.db',
                          direction='maximize',
                          load_if_exists=True)

[I 2026-03-15 22:16:40,295] A new study created in RDB with name: xgboost_fingerprints_big


In [86]:
study.optimize(objective, n_trials=10)

/tmp/ipykernel_471808/3326202407.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  gamma = trial.suggest_loguniform('gamma', 0.01, 10)
/tmp/ipykernel_471808/3326202407.py:7: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  colsample_bytree = trial.suggest_uniform('colsample_bytree', 0.1, 0.9)
[I 2026-03-15 22:17:07,350] Trial 0 finished with value: 1.0 and parameters: {'max_depth': 5, 'gamma': 0.013837560607443624, 'colsample_bytree': 0.846783286019613}. Best is trial 0 with value: 1.0.
[I 2026-03-15 22:17:23,881] Trial 1 finished with value: 1.0 and parameters: {'max_depth': 5, 'gamma': 0.12860138766285287, 'colsample_bytree': 0.21791859661345292}. Best is trial 0 with value: 1.0.
[I 202

In [87]:
study.best_params

{'max_depth': 5,
 'gamma': 0.013837560607443624,
 'colsample_bytree': 0.846783286019613}

In [88]:
study.best_value

1.0

In [89]:
with open('XGB_best_params_big.pkl', 'wb') as file:
    pickle.dump(study, file)

In [90]:
with open('XGB_best_params_big.pkl', 'rb') as file:
    study = pickle.load(file)

In [91]:
study

In [92]:
from sklearn.model_selection import train_test_split

X_t, X_val, y_t, y_val = train_test_split(X_train, y_train, test_size=0.2)

dt = xgb.DMatrix(X_t, label=y_t)
dval = xgb.DMatrix(X_val, label=y_val)

params = {
          'max_depth': study.best_params['max_depth'],
          'eta': 0.05,
          'objective': 'binary:logistic',
          'eval_metric':['auc', 'logloss', 'map'],
          'tree_method': 'hist',
          'gamma': study.best_params['gamma'],
          'colsample_bytree': study.best_params['colsample_bytree'],
          'subsample': 0.5,
          'nthread': 8,
          'base_score': prevalence
}

bst = xgb.train(params,
                dt,
                num_boost_round=500,
                verbose_eval=10,
                early_stopping_rounds=20,
                evals=[(dtrain, 'train'), (dval, 'val')])

bst.save_model('generalXGB_big.model')

[0]	train-auc:0.98519	train-logloss:0.39392	train-map:1.00000	val-auc:0.98596	val-logloss:0.39189	val-map:0.96777
[10]	train-auc:0.99707	train-logloss:0.20212	train-map:1.00000	val-auc:0.99709	val-logloss:0.20097	val-map:1.00000
[20]	train-auc:0.99814	train-logloss:0.12584	train-map:1.00000	val-auc:0.99766	val-logloss:0.12512	val-map:1.00000
[21]	train-auc:0.99823	train-logloss:0.12052	train-map:1.00000	val-auc:0.99767	val-logloss:0.11986	val-map:1.00000


/tmp/ipykernel_471808/2525310426.py:28: UserWarning: [22:21:43] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  bst.save_model('generalXGB_big.model')


In [93]:
test_df['ROMol'] = test_df['canonical_smiles'].progress_apply(Chem.MolFromSmiles)
test_df['MorganFP'] = test_df['ROMol'].parallel_apply(calc_morgan_fingerprints)

  0%|          | 0/6942 [00:00<?, ?it/s]

/tmp/ipykernel_471808/3682329790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['ROMol'] = test_df['canonical_smiles'].progress_apply(Chem.MolFromSmiles)


/tmp/ipykernel_471808/3682329790.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['MorganFP'] = test_df['ROMol'].parallel_apply(calc_morgan_fingerprints)


In [94]:
X_test = csr_matrix(np.vstack(test_df['MorganFP'].values))

In [95]:
y_test = test_df['class'].values

In [96]:
test_df['class'] = test_df['class'].astype(int)

/tmp/ipykernel_471808/499460510.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['class'] = test_df['class'].astype(int)


In [97]:
from sklearn.model_selection import train_test_split

#X_t, X_val, y_t, y_val = train_test_split(X_train, y_train, test_size=0.2)

dt = xgb.DMatrix(X_train, label=y_train)
#dval = xgb.DMatrix(X_test, label=y_test)

params = {
          'max_depth': study.best_params['max_depth'],
          'eta': 0.05,
          'objective': 'binary:logistic',
          'eval_metric':['auc', 'logloss', 'map'],
          'tree_method': 'hist',
          'gamma': study.best_params['gamma'],
          'colsample_bytree': study.best_params['colsample_bytree'],
          'subsample': 0.5,
          'nthread': 8,
          'base_score': prevalence
}

bst = xgb.train(params,
                dt,
                num_boost_round=500,
                verbose_eval=10)

bst.save_model('general_all_XGB_big.model')

/tmp/ipykernel_471808/332217036.py:26: UserWarning: [22:22:57] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  bst.save_model('general_all_XGB_big.model')


In [98]:
test_df['ROMol'] = test_df['canonical_smiles'].progress_apply(Chem.MolFromSmiles)
test_df['MorganFP'] = test_df['ROMol'].parallel_apply(calc_morgan_fingerprints)

  0%|          | 0/6942 [00:00<?, ?it/s]

/tmp/ipykernel_471808/3682329790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['ROMol'] = test_df['canonical_smiles'].progress_apply(Chem.MolFromSmiles)


/tmp/ipykernel_471808/3682329790.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['MorganFP'] = test_df['ROMol'].parallel_apply(calc_morgan_fingerprints)


In [99]:
X_test = csr_matrix(np.vstack(test_df['MorganFP'].values))

In [100]:
y_test = test_df['class'].values

In [101]:
from sklearn.metrics import roc_auc_score, precision_score, confusion_matrix, f1_score

dtest = xgb.DMatrix(X_test)
y_pred = bst.predict(dtest)
auc = roc_auc_score(y_test, y_pred)
print(f'auc is {auc:.5f}')

for th in np.arange(0.0, 1.0, 0.01):
    y_pred_binary = (y_pred > th).astype(int)
    cm = confusion_matrix(y_test, y_pred_binary)
    tp, fp, fn, tn = cm[1, 1], cm[0, 1], cm[1, 0], cm[0, 0]
    precision = precision_score(y_test, y_pred > th)
    f1 = f1_score(y_test, y_pred > th)
    print(f'th: {th:.3f}\t tp: {tp}\t tn: {tn}\tfp: {fp}\tfn: {fn}\t prec: {precision:.4f} \t f1_score: {f1:.4f}')

auc is 0.99899
th: 0.000	 tp: 999	 tn: 0	fp: 5943	fn: 0	 prec: 0.1439 	 f1_score: 0.2516
th: 0.010	 tp: 996	 tn: 5842	fp: 101	fn: 3	 prec: 0.9079 	 f1_score: 0.9504
th: 0.020	 tp: 996	 tn: 5884	fp: 59	fn: 3	 prec: 0.9441 	 f1_score: 0.9698
th: 0.030	 tp: 995	 tn: 5897	fp: 46	fn: 4	 prec: 0.9558 	 f1_score: 0.9755
th: 0.040	 tp: 995	 tn: 5902	fp: 41	fn: 4	 prec: 0.9604 	 f1_score: 0.9779
th: 0.050	 tp: 995	 tn: 5907	fp: 36	fn: 4	 prec: 0.9651 	 f1_score: 0.9803
th: 0.060	 tp: 995	 tn: 5910	fp: 33	fn: 4	 prec: 0.9679 	 f1_score: 0.9817
th: 0.070	 tp: 995	 tn: 5914	fp: 29	fn: 4	 prec: 0.9717 	 f1_score: 0.9837
th: 0.080	 tp: 995	 tn: 5914	fp: 29	fn: 4	 prec: 0.9717 	 f1_score: 0.9837
th: 0.090	 tp: 995	 tn: 5914	fp: 29	fn: 4	 prec: 0.9717 	 f1_score: 0.9837
th: 0.100	 tp: 995	 tn: 5916	fp: 27	fn: 4	 prec: 0.9736 	 f1_score: 0.9847
th: 0.110	 tp: 995	 tn: 5916	fp: 27	fn: 4	 prec: 0.9736 	 f1_score: 0.9847
th: 0.120	 tp: 995	 tn: 5918	fp: 25	fn: 4	 prec: 0.9755 	 f1_score: 0.9856
th: 0.130	

In [102]:
with open('XGB_best_model_big.pkl', 'wb') as file:
    pickle.dump(bst, file)

# thr = 0.970
# thr = 0.500 (bad precision = 0.9890)
# 0.640

# Classic Random Forest model

In [104]:
import optuna
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 1, 100)
    max_depth = trial.suggest_int('max_depth', 2, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 4, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 4, 20)

    # Create a RandomForest model
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    # Perform cross validation
    scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='roc_auc')

    # Optuna aims to minimize the objective, hence we negate the accuracy
    return np.mean(scores)

In [105]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-03-15 22:25:58,498] A new study created in memory with name: no-name-11b1eef8-3933-44b3-9c9d-7ed2d76ee0a8
[I 2026-03-15 22:26:00,429] Trial 0 finished with value: 0.9871548986127431 and parameters: {'n_estimators': 11, 'max_depth': 19, 'min_samples_split': 20, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.9871548986127431.
[I 2026-03-15 22:26:12,801] Trial 1 finished with value: 0.990717087221476 and parameters: {'n_estimators': 77, 'max_depth': 25, 'min_samples_split': 6, 'min_samples_leaf': 6}. Best is trial 1 with value: 0.990717087221476.
[I 2026-03-15 22:26:19,181] Trial 2 finished with value: 0.9906169428112046 and parameters: {'n_estimators': 46, 'max_depth': 11, 'min_samples_split': 12, 'min_samples_leaf': 7}. Best is trial 1 with value: 0.990717087221476.
[I 2026-03-15 22:26:33,337] Trial 3 finished with value: 0.990236973866252 and parameters: {'n_estimators': 94, 'max_depth': 16, 'min_samples_split': 9, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.9

In [106]:
study.best_params

{'n_estimators': 93,
 'max_depth': 10,
 'min_samples_split': 14,
 'min_samples_leaf': 6}

In [107]:
with open('RF_best_params_big.pkl', 'wb') as file:
    pickle.dump(study, file)

In [108]:
from sklearn.model_selection import train_test_split

#X_t, X_val, y_t, y_val = train_test_split(X_train, y_train, test_size=0.2)

#dt = xgb.DMatrix(X_train, label=y_train)
#dval = xgb.DMatrix(X_test, label=y_test)

params = {
          'n_estimators': study.best_params['n_estimators'],
          'max_depth': study.best_params['max_depth'],
        'min_samples_split': study.best_params['min_samples_split'],
        'min_samples_leaf': study.best_params['min_samples_leaf'],
        'random_state': 42
}

bst = RandomForestClassifier(**params).fit(X_train, y_train)

#scores = cross_val_score(bst, X_train, y_train, cv=5, scoring='accuracy')

In [109]:
X_test = csr_matrix(np.vstack(test_df['MorganFP'].values))

In [110]:
y_test = test_df['class'].values

In [111]:
bst

RandomForestClassifier(max_depth=10, min_samples_leaf=6, min_samples_split=14,
                       n_estimators=93, random_state=42)

In [112]:
from sklearn.metrics import roc_auc_score, precision_score, confusion_matrix, f1_score

y_pred = bst.predict(X_test)
auc = roc_auc_score(y_test, y_pred)
print(f'auc is {auc:.5f}')

y_pred = bst.predict_proba(X_test)[:, 1]
for th in np.arange(0.0, 1.0, 0.01):
    y_pred_binary = (y_pred > th).astype(int)
    cm = confusion_matrix(y_test, y_pred_binary)
    tp, fp, fn, tn = cm[1, 1], cm[0, 1], cm[1, 0], cm[0, 0]
    precision = precision_score(y_test, y_pred > th)
    f1 = f1_score(y_test, y_pred > th)
    print(f'th: {th:.3f}\t tp: {tp}\t tn: {tn}\tfp: {fp}\tfn: {fn}\t prec: {precision:.4f} \t f1_score: {f1:.4f}')

auc is 0.98506
th: 0.000	 tp: 999	 tn: 0	fp: 5943	fn: 0	 prec: 0.1439 	 f1_score: 0.2516
th: 0.010	 tp: 998	 tn: 3292	fp: 2651	fn: 1	 prec: 0.2735 	 f1_score: 0.4294
th: 0.020	 tp: 998	 tn: 4613	fp: 1330	fn: 1	 prec: 0.4287 	 f1_score: 0.5999
th: 0.030	 tp: 996	 tn: 5187	fp: 756	fn: 3	 prec: 0.5685 	 f1_score: 0.7241
th: 0.040	 tp: 994	 tn: 5484	fp: 459	fn: 5	 prec: 0.6841 	 f1_score: 0.8108
th: 0.050	 tp: 993	 tn: 5645	fp: 298	fn: 6	 prec: 0.7692 	 f1_score: 0.8672
th: 0.060	 tp: 992	 tn: 5751	fp: 192	fn: 7	 prec: 0.8378 	 f1_score: 0.9088
th: 0.070	 tp: 992	 tn: 5806	fp: 137	fn: 7	 prec: 0.8787 	 f1_score: 0.9323
th: 0.080	 tp: 992	 tn: 5836	fp: 107	fn: 7	 prec: 0.9026 	 f1_score: 0.9457
th: 0.090	 tp: 992	 tn: 5862	fp: 81	fn: 7	 prec: 0.9245 	 f1_score: 0.9575
th: 0.100	 tp: 990	 tn: 5877	fp: 66	fn: 9	 prec: 0.9375 	 f1_score: 0.9635
th: 0.110	 tp: 987	 tn: 5892	fp: 51	fn: 12	 prec: 0.9509 	 f1_score: 0.9691
th: 0.120	 tp: 987	 tn: 5898	fp: 45	fn: 12	 prec: 0.9564 	 f1_score: 0.9719

In [113]:
with open('RF_best_model_big.pkl', 'wb') as file:
    pickle.dump(bst, file)

In [117]:
test_df['class'].value_counts()

class
0    5943
1     999
Name: count, dtype: int64

In [115]:
# thr = 0.970
# thr = 0.5

In [ ]:
#